### Pinecone Hybrid Search Integration

Pinecone is a vector database with broad functionality.

In [7]:
#importing libraries

import os
from dotenv import load_dotenv

from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder

from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [2]:
index_name = "langchain-pinecone-hybrid-search"

#initializing pinecone client
pc = Pinecone(api_key=api_key)

#creating the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536, #dimensionality of dense model
        metric="dotproduct", #sparse values supported only for dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

In [3]:
index = pc.Index(index_name)
index

In [ ]:
#getting embeddings (used for dense vector) 
embedding = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

In [9]:
#to encode text to sparse vector , we can use SPLADE or BM25, for out of domain tasks we can use BM25.
# it use tf-idf values

bm25_encoder = BM25Encoder().default()

sentences= ["this is good course", "im learning agentic ai", "langchain is very useful to use llms"]

#fitting tf-idf values on corpus

bm25_encoder.fit(sentences)

#store the values to json file
bm25_encoder.dump("bm25_values.json")

#loading bm25_encoder object
bm25_encoder_obj = BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<?, ?it/s]


In [ ]:
#constructing retriever